In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader


In [2]:
DATA_ROOT = Path(
    "/home/rittick-dutta/Desktop/Research/"
    "symile-mimic/symile-mimic-aws/data_npy"
)

In [3]:
class SymileMIMICDataset(Dataset):
    def __init__(self, data_root, split="train"):
        self.split_dir = Path(data_root) / split

        if not self.split_dir.is_dir():
            raise FileNotFoundError(self.split_dir)

        def load_array(name):
            path = self.split_dir / f"{name}_{split}.npy"
            return np.load(path, mmap_mode="r", allow_pickle=False)

        self.cxr = load_array("cxr")
        self.ecg = load_array("ecg")
        self.labs_percentiles = load_array("labs_percentiles")

        # 1 means observed; 0 means missing.
        self.labs_observed = load_array("labs_missingness")
        self.hadm_ids = load_array("hadm_id")

        n = len(self.cxr)

        for name, array in {
            "ecg": self.ecg,
            "labs_percentiles": self.labs_percentiles,
            "labs_observed": self.labs_observed,
            "hadm_id": self.hadm_ids,
        }.items():
            if len(array) != n:
                raise ValueError(
                    f"{name} has {len(array)} samples; expected {n}."
                )

        if self.labs_percentiles.shape != (n, 50):
            raise ValueError("Expected lab percentiles with shape (N, 50).")

        if self.labs_observed.shape != (n, 50):
            raise ValueError("Expected lab indicators with shape (N, 50).")

    def __len__(self):
        return len(self.cxr)

    def __getitem__(self, index):
       
        cxr = torch.from_numpy(
            np.array(self.cxr[index], dtype=np.float32, copy=True)
        )
        ecg = torch.from_numpy(
            np.array(self.ecg[index], dtype=np.float32, copy=True)
        )
        labs_percentiles = torch.from_numpy(
            np.array(
                self.labs_percentiles[index],
                dtype=np.float32,
                copy=True,
            )
        )
        labs_observed = torch.from_numpy(
            np.array(
                self.labs_observed[index],
                dtype=np.float32,
                copy=True,
            )
        )

        # 50 percentile values followed by 50 availability indicators.
        labs = torch.cat([labs_percentiles, labs_observed], dim=0)

        return {
            "cxr": cxr,
            "ecg": ecg,
            "labs": labs,
            "hadm_id": torch.tensor(
                int(self.hadm_ids[index]), dtype=torch.long
            ),
        }


In [4]:
train_dataset = SymileMIMICDataset(DATA_ROOT, split="train")
val_dataset = SymileMIMICDataset(DATA_ROOT, split="val")

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))

Training samples: 10000
Validation samples: 750


In [5]:
sample = train_dataset[0]

print("Single sample:")
for name, tensor in sample.items():
    print(f"{name}: shape={tuple(tensor.shape)}, dtype={tensor.dtype}")

batch = next(iter(train_loader))

print("\nBatch:")
for name, tensor in batch.items():
    print(f"{name}: shape={tuple(tensor.shape)}, dtype={tensor.dtype}")

Single sample:
cxr: shape=(3, 320, 320), dtype=torch.float32
ecg: shape=(1, 5000, 12), dtype=torch.float32
labs: shape=(100,), dtype=torch.float32
hadm_id: shape=(), dtype=torch.int64

Batch:
cxr: shape=(8, 3, 320, 320), dtype=torch.float32
ecg: shape=(8, 1, 5000, 12), dtype=torch.float32
labs: shape=(8, 100), dtype=torch.float32
hadm_id: shape=(8,), dtype=torch.int64
